## Lecture 4: Introduction to machine learning

* Data pre-processing
* Classification
* Regression
* Clustering
* Model evaluation

Learning objectives:
* Understand the concept of machine learning and prediction
* Understand different types of machine learning
* Implement a simple machine learning workflow
* Evaluate model performance

Recommended reading from the course materials: Chapter 7 from Introduction: Data Science & Analytics in Finance, Economics, and Business by Mikko Ranta
[link](https://mranta-ai.github.io/data-science-in-finance/).

1. Supervised Learning - Classification: Predicting discrete financial states such as corporate distress, credit default, or audit risk

2. Supervised Learning - Regression: Estimating continuous financial targets like retun on assets, return volatility, or firm valuation

3. Unsupervised Learning - Clustering: Partitioning unlabelled corporate financial ratios into risk and operational profiles

4. Model Evaluation & Diagnostics: Analyzing bias-variance tradeoffs, stratified cross-validation, confusion matrices, ROC-AUC curves, Precision-Recall curves, partial dependence displays, and residual error analysis.

[Video about machine learning](https://www.youtube.com/watch?v=f_uwKZIAeM0)

Machine learning workflow
1. Load data into your notebook and inspect your data
2. Execute necessary data processing steps, such as removing missing values and duplicates, and dealing with outliers
3. Create calculated variables, such as log-transformations, and binary variables, standardise variables if needed
4. Define your X and Y variables and split your data into training and testing samples
5. Execute the machine learning model
6. Evaluate the model performance
7. Start over to improve the model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.display.float_format = "{:,.2f}".format

In [2]:
df = pd.read_excel("/home/jovyan/shared/Data/time_series_lecture.xlsx", index_col=0)
df

,Company name Latin alphabet,Inactive,Quoted,Branch,OwnData,Woco,Country ISO code,"NACE Rev. 2, core code (4 digits)",Consolidation code,Last avail. year,...,Stock EUR 2020,Stock EUR 2019,Stock EUR 2018,Stock EUR 2017,Costs of goods sold EUR 2022,Costs of goods sold EUR 2021,Costs of goods sold EUR 2020,Costs of goods sold EUR 2019,Costs of goods sold EUR 2018,Costs of goods sold EUR 2017
1.00,KONE OYJ,No,Yes,No,No,Yes,FI,2822,C1,2025,...,597000000,648600000,624100000,626800000,4458200000,4297300000,3957600000,4022300000,3614400000,3399900000
2.00,WARTSILA OYJ,No,Yes,No,No,Yes,FI,2815,C2,2025,...,1192000000,1365000000,1165000000,1051000000,3695000000,2710000000,2655000000,2866000000,2788000000,2533000000
3.00,METSO OYJ,No,Yes,No,No,Yes,FI,2892,C1,2025,...,1038000000,975000000,208900000,195900000,3893000000,3089000000,2429000000,2077000000,1067700000,859800000
4.00,VALMET OYJ,No,Yes,No,No,Yes,FI,2895,C1,2025,...,553000000,514000000,419000000,415000000,3799000000,2899000000,2800000000,2641000000,2525000000,2323000000
5.00,KONECRANES OYJ,No,Yes,No,No,Yes,FI,2822,C1,2025,...,644800000,658700000,635800000,545000000,1096600000,995700000,1072500000,1104200000,1053300000,1148900000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"1,108.00",POME OY,No,No,No,No,No,FI,2892,U1,2025,...,685000,768000,756000,766000,n.a.,n.a.,n.a.,2260000,2284000,n.a.
"1,109.00",ULVILAN KONEPAJA OY,No,No,No,No,No,FI,2892,U1,2025,...,102000,200000,282000,240000,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.
"1,110.00",KEIMOLAN KONE OY,No,No,No,No,No,FI,2899,U1,2025,...,0,0,0,0,n.a.,n.a.,n.a.,n.a.,2000,2000
"1,111.00",VIESKAN METALLI OY,No,No,No,No,No,FI,2830,U1,2024,...,0,0,0,0,n.a.,n.a.,n.a.,n.a.,n.a.,n.a.


Fast forward the processing steps from last lecture.

In [3]:
id_cols = [
    'Company name Latin alphabet',
    'Country ISO code',
    'NACE Rev. 2, core code (4 digits)',
    'Consolidation code'
]

variables = {
    "ROA": "ROA using Profit (Loss) before tax",
    "Revenue": "Operating revenue (Turnover) EUR",
    "Assets": "Total assets EUR",
    "CurrentRatio": "Current ratio",
    "LongTermDebt": "Long term debt EUR",
    "ShortTermDebt": "Loans & short-term debt EUR",
    "AccountsReceivable": "Debtors EUR",
    "AccountsPayable": "Creditors EUR",
    "Inventory": "Stock EUR",
    "COGS": "Costs of goods sold EUR"
}

#Not required for the assignment data

panel_list = []

for year in range(2017, 2023):

    temp = df[id_cols].copy()

    temp["Year"] = year

    for short_name, prefix in variables.items():

        col = f"{prefix} {year}"

        if col in df.columns:
            temp[short_name] = df[col]

    panel_list.append(temp)

panel_df = pd.concat(panel_list, ignore_index=True)

panel_df

,Company name Latin alphabet,Country ISO code,"NACE Rev. 2, core code (4 digits)",Consolidation code,Year,ROA,Revenue,Assets,CurrentRatio,LongTermDebt,ShortTermDebt,AccountsReceivable,AccountsPayable,Inventory,COGS
0,KONE OYJ,FI,2822,C1,2017,16.76,8820500000,7461900000,1.33,194700000,40400000,1910800000,705100000,626800000,3399900000
1,WARTSILA OYJ,FI,2815,C2,2017,8.69,4968000000,5648000000,1.41,517000000,101000000,1307000000,539000000,1051000000,2533000000
2,METSO OYJ,FI,2892,C1,2017,1.18,1145400000,1356800000,1.40,183500000,45000000,187700000,99500000,195900000,859800000
3,VALMET OYJ,FI,2895,C1,2017,5.43,3074000000,2908000000,1.02,201000000,18000000,546000000,287000000,415000000,2323000000
4,KONECRANES OYJ,FI,2822,C1,2017,7.74,3363600000,3562900000,1.26,600800000,49800000,538200000,201200000,545000000,1148900000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6667,POME OY,FI,2892,U1,2022,16.27,n.a.,1899000,1.99,292000,286000,408000,248000,862000,n.a.
6668,ULVILAN KONEPAJA OY,FI,2892,U1,2022,6.88,n.a.,3226000,0.99,41000,1264000,1928000,795000,503000,n.a.
6669,KEIMOLAN KONE OY,FI,2899,U1,2022,n.a.,n.a.,4000,n.s.,n.a.,0,0,0,0,n.a.
6670,VIESKAN METALLI OY,FI,2830,U1,2022,-60,n.a.,5000,n.s.,n.a.,0,0,0,0,n.a.


In [4]:
#remove duplicates
panel_df.drop_duplicates(inplace = True)
#Fix missing values
panel_df.replace({"n.a.": None, "n.s.": None}, inplace=True)
#numeric columns
num_cols = [  'ROA', 'Revenue', 'Assets', 'CurrentRatio', 'LongTermDebt',
       'ShortTermDebt', "Inventory", 'AccountsPayable', 'AccountsReceivable',
       'COGS']

panel_df[num_cols] = panel_df[num_cols].astype('float64')
#Drop missing values
panel_df.dropna(inplace=True)

panel_df

,Company name Latin alphabet,Country ISO code,"NACE Rev. 2, core code (4 digits)",Consolidation code,Year,ROA,Revenue,Assets,CurrentRatio,LongTermDebt,ShortTermDebt,AccountsReceivable,AccountsPayable,Inventory,COGS
0,KONE OYJ,FI,2822,C1,2017,16.76,"8,820,500,000.00","7,461,900,000.00",1.33,"194,700,000.00","40,400,000.00","1,910,800,000.00","705,100,000.00","626,800,000.00","3,399,900,000.00"
1,WARTSILA OYJ,FI,2815,C2,2017,8.69,"4,968,000,000.00","5,648,000,000.00",1.41,"517,000,000.00","101,000,000.00","1,307,000,000.00","539,000,000.00","1,051,000,000.00","2,533,000,000.00"
2,METSO OYJ,FI,2892,C1,2017,1.18,"1,145,400,000.00","1,356,800,000.00",1.40,"183,500,000.00","45,000,000.00","187,700,000.00","99,500,000.00","195,900,000.00","859,800,000.00"
3,VALMET OYJ,FI,2895,C1,2017,5.43,"3,074,000,000.00","2,908,000,000.00",1.02,"201,000,000.00","18,000,000.00","546,000,000.00","287,000,000.00","415,000,000.00","2,323,000,000.00"
4,KONECRANES OYJ,FI,2822,C1,2017,7.74,"3,363,600,000.00","3,562,900,000.00",1.26,"600,800,000.00","49,800,000.00","538,200,000.00","201,200,000.00","545,000,000.00","1,148,900,000.00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6597,STECOS OY,FI,2829,U1,2022,14.44,"1,614,000.00","575,000.00",1.52,"163,000.00","43,000.00","60,000.00","108,000.00","387,000.00","742,000.00"
6615,POHJOLAN KONEPAJA OY,FI,2829,U1,2022,1.00,"135,000.00","199,000.00",1.59,"50,000.00","20,000.00","9,000.00","50,000.00",0.00,"71,000.00"
6627,KP-ILMA OY,FI,2825,U1,2022,16.78,"1,900,000.00","906,000.00",2.38,"157,000.00",0.00,"224,000.00","69,000.00","13,000.00","1,234,000.00"
6630,KTS-RAKENNE OY,FI,2811,U1,2022,85.71,"54,000.00","14,000.00",0.01,"67,000.00",0.00,0.00,0.00,0.00,"2,000.00"


In [5]:
#Descriptive stats to check that the data looks correct
panel_df.describe()

,"NACE Rev. 2, core code (4 digits)",Year,ROA,Revenue,Assets,CurrentRatio,LongTermDebt,ShortTermDebt,AccountsReceivable,AccountsPayable,Inventory,COGS
count,"2,914.00","2,914.00","2,914.00","2,914.00","2,914.00","2,914.00","2,914.00","2,914.00","2,914.00","2,914.00","2,914.00","2,914.00"
mean,"2,853.95","2,019.55",5.87,"87,155,983.59","82,477,544.22",2.35,"7,483,219.20","1,654,228.81","14,209,023.12","8,977,726.70","14,654,749.83","49,319,455.64"
std,35.11,1.70,18.42,"598,803,767.11","596,312,995.76",3.01,"67,422,962.24","12,304,980.26","124,534,491.51","65,189,982.32","95,838,405.69","307,912,592.36"
min,"2,811.00","2,017.00",-97.47,"-16,000.00",127.00,0.01,"-5,000.00",0.00,"-8,000.00","-12,000.00",0.00,"-35,000.00"
25%,"2,825.00","2,018.00",0.08,"477,000.00","379,250.00",1.19,0.00,0.00,"26,000.00","19,558.75","42,000.00","172,000.00"
50%,"2,830.00","2,020.00",6.10,"2,276,000.00","1,572,500.00",1.68,"60,361.50","23,581.00","216,500.00","127,000.00","358,000.00","1,106,500.00"
75%,"2,895.00","2,021.00",14.67,"11,270,485.75","7,579,228.25",2.65,"362,250.00","161,739.75","1,287,500.00","774,750.00","2,002,000.00","6,067,355.50"
max,"2,899.00","2,022.00",87.88,"10,951,700,000.00","9,720,400,000.00",85.00,"1,235,000,000.00","264,500,000.00","2,668,100,000.00","1,310,200,000.00","1,846,000,000.00","4,458,200,000.00"


## Minimal machine learning implementation 

Using Linear Discriminant analysis

You can use this for any other classification model as well if you change the model.

You will survive assignment 4 if you implement a similar workflow to this.


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis #import the model   


#Note that ROA can't be in both X and Y: you can't predict ROA with ROA
X = panel_df[[
        "Revenue",
        "Assets",
        "CurrentRatio",
        "LongTermDebt",
        "ShortTermDebt",
        "Inventory",
        "AccountsPayable",
        "AccountsReceivable",
        "COGS"]]
y = panel_df["ROA"] > 0

#Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42 #define random_state for replicable results, optional
)

#Initialize the machine learning model
model = LinearDiscriminantAnalysis() 

#Train the model
model.fit(X_train, y_train,seed=42)

model.score(X_test,y_test)

TypeError: LinearDiscriminantAnalysis.fit() got an unexpected keyword argument 'seed'

## Now let's take a deeper look into machine learning and prediction

We will try to predict ROA using working capital ratios and a set of basic financial markers

Calculate variables

* Revenue change
* Log assets
* Leverage
* Cash conversion cycle
* Accounts payable turnover (days)
* Inventory turnover (days)
* Accounts receivable turnover (days)

Already in the data
* Current ratio
* ROA


In [ ]:
panel_df["delta_revenue"] = panel_df.groupby("Company name Latin alphabet")["Revenue"].diff()

In [ ]:
panel_df["log_assets"] = np.log1p(panel_df["Assets"])

In [ ]:
panel_df["leverage"] = (panel_df["ShortTermDebt"]+panel_df["LongTermDebt"])/panel_df["Assets"]*100

In [ ]:
panel_df["inventory_turnover"] = panel_df["Inventory"]/panel_df["COGS"]*365
panel_df["accounts_receivable_turnover"] = panel_df["AccountsReceivable"]/panel_df["Revenue"]*365
panel_df["accounts_payable_turnover"] = panel_df["AccountsPayable"]/panel_df["COGS"]*365

#make sure your new variables are float
panel_df[["inventory_turnover", "accounts_receivable_turnover", "accounts_payable_turnover"]] = panel_df[["inventory_turnover", "accounts_receivable_turnover", "accounts_payable_turnover"]].astype("float64")

#Cash conversion cycle
panel_df["CCC"] = panel_df["inventory_turnover"]+panel_df["accounts_receivable_turnover"]-panel_df["accounts_payable_turnover"]

In [ ]:
desc_cols = num_cols+[ "delta_revenue","log_assets", "leverage", "inventory_turnover",
                      "accounts_receivable_turnover", "accounts_payable_turnover", "CCC"]
#Winsorize
panel_df[desc_cols] = panel_df[desc_cols].clip(lower=panel_df[desc_cols].quantile(0.025), upper=panel_df[desc_cols].quantile(0.975), axis=1)

In [ ]:
#Check the datatypes
panel_df.info()

In [ ]:
#Plot histogram to see the distributions
panel_df[desc_cols].hist(figsize=(14,14),grid=False, edgecolor='black')    # Plot the histograms
plt.show()

Some outliers still exist but winsorization has removed the most problematic outliers. Using variables at very different scales may bias the results or lead to certain variables affecting the results disproportionately. Standardization can also be used to improve the distribution of the variables.

The distribution should look something like this:

<img src="https://upload.wikimedia.org/wikipedia/commons/9/9e/Normal_Distribution_NIST.gif?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=original" alt="Normal distribution">

Next we need to check the correlations because highly correlated variables will cause problems with linear models.

In [ ]:
panel_df[desc_cols].corr()

In [ ]:
corr = panel_df[desc_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
desc_cols

You can always start by seeing what happens if you include all possible variables and then start shrinking.

Exclude highly correlated variables and don't include both the log- and non-log-version of a variable.

Exclude revenue and assets and use change in revenue (delta_revenue) and logged assets instead.

Working capital turnover variables instead of the monetary values.

Instead of long- and short-term debt we use leverage.

Including a variable which is a direct sum of other included variables might result in problems. (this time we don't have one)

In [ ]:
#Final set of variables for prediction
y_var = [ 'ROA']

#Can working capital predict ROA?
pred_vars = [ 'CurrentRatio',
              'leverage',
             'delta_revenue',
             'log_assets',
             'inventory_turnover',
             'accounts_receivable_turnover',
             'accounts_payable_turnover',
             'CCC']

In [ ]:
corr = panel_df[pred_vars].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
#Drop NA values

panel_df.dropna(subset=pred_vars, inplace=True)
panel_df

Predicting ROA using classification and then regression
1. For classification we will try to predict whether ROA is positive or not, thus we create a new ROA > 1 variable
2. For regression we will try to predict ROA as a discrete variable

In [ ]:
# Create the binary target variable for classification
panel_df['ROA_ind'] = panel_df[y_var] > 0
panel_df['ROA_ind'].sum()

In [ ]:
# Prepare feature matrices and target vectors

#For classification models
X_class = panel_df[pred_vars]
y_class = panel_df['ROA_ind']

#For regression models
X_reg = panel_df[pred_vars]
y_reg = panel_df['ROA']

print(f"Classification features shape: {X_class.shape}")
print(f"Regression features shape: {X_reg.shape}")

Splitting the data into training and testing parts for classification.
The model will use the training part to learn the connections and dependencies without seeing the testing data. The testing data will be used to see, how the model performs on new data. 

We will use 80% of the data for training and 20% for testing.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Split data into training and test sets
X_train, X_test , y_train, y_test = train_test_split(X_class, y_class, test_size=0.2, random_state=1)

Linear discriminant analysis

Linear Discriminant Analysis (LDA) is one of the simplest classification techniques, as it aims to separate classes using a linear combination of features. It can be easily applied to both binary and multi-class classification problems.

Imagine plotting observations using several variables and then drawing a new axis that places members of the same group close together while pushing different groups as far apart as possible.

![](https://mranta-ai.github.io/data-science-in-finance/build/8298097b7f523e76a46cef7b8432e8f8.png)

Picture source: https://mranta-ai.github.io/data-science-in-finance/build/8298097b7f523e76a46cef7b8432e8f8.png

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis    # Import the model

LDA_class = LinearDiscriminantAnalysis()    # Create the model

In [ ]:
LDA_class.fit(X_train,y_train)  # Fit the model to the data

In [ ]:
LDA_class.score(X_test,y_test)  # Calculate the accuracy of the model

In [ ]:
#Let's use classification_report to get multiple accuracy measures easily
from sklearn.metrics import classification_report

## Model evaluation

Imagine a coin flip as the baseline prediction with two predicted categories. With a coin flip you have a 50 % chance of getting your prediction right.

At the very least you want your prediction accuracy to be better than coin flip, so higher than 50 %. If the accuracy is less than 50 % you can flip the categories and have again performance > 50 %.

Evaluation metrics

* Accuracy = correct classifications / total classifications
* Recall = correctly classified actual positives / all **actual** positives
* Precision = correctly classified actual positives / all observations **classified** as positive
* F1 = mean of precision and recall = (precision x recall) / (precision+recall) x2


In [ ]:
#Using the testing data
#True = ROA > 0 False = ROA <= 0

print(classification_report(y_test,LDA_class.predict(X_test)))

In [ ]:
from sklearn.metrics import log_loss
from sklearn.metrics import RocCurveDisplay    # Import the RocCurveDisplay function
from sklearn.metrics import PrecisionRecallDisplay    # Import the PrecisionRecallDisplay function

Log loss is a measure of model confidence and correctness. Log loss is 0 for a perfect model and gets higher for a poor prediction. It is not limited to a maximum of 1.

Model predicts ROA > 0 correctly with 99% confidence -> low log loss

Model predicts ROA > 0 correctly with 60% confidence -> a bit higher log loss

Model predicts ROA > 0 incorrectly with 99% confidence -> very high log loss

In [ ]:
log_loss(y_test,LDA_class.predict_proba(X_test))

ROC and AUC

Representing model performance across all possible thresholds. For example, the model gives one observation 99% chance of having ROA > 0 and another 51% chance of having ROA > 0. If the threshold is 0.5, both will be classified as ROA > 0, but if the threshold is 0.75, only the 99% observation is classified as ROA > 0.

ROC (Receiver-operating characteristic) and AUC for a perfect model

![](https://developers.google.com/static/machine-learning/crash-course/images/auc_1-0.png)

ROC and AUC for a coin flip

![](https://developers.google.com/static/machine-learning/crash-course/images/auc_0-5.png)

Source: https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc



In [ ]:
RocCurveDisplay.from_estimator(LDA_class,X_test,y_test)   # Plot the ROC curve
plt.show()

Precision-Recall curve

* Recall = correctly classified actual positives / all **actual** positives
* Precision = correctly classified actual positives / all observations **classified** as positive

For example: recall for the True category would be 1 if the model predicts all observations as True but the more observations are classified as True, the more False observations can be incorrectly classified.

![](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*6QPLsDvjo4H6OZrxEBI8Fg.png)

Source: https://medium.com/@douglaspsteen/precision-recall-curves-d32e5b290248

In [ ]:
PrecisionRecallDisplay.from_estimator(LDA_class,X_test,y_test)  # Plot the precision-recall curve
plt.show()

K-Nearest Neigbours (KNN)

* KNN classifies new observations based on their similarity with existing data points
* Similarity can be measured in various ways, with cosine similarity being one common approach.
* KNN is particularly useful when there is limited knowledge about the underlying model that could describe an accounting dataset.

  -> In such cases, the best strategy is to classify a new instance based on the class of the most similar instances in the training data

K-parameter

The algorithm is highly sensitive to the K parameter, which determines the number of nearest neighbors considered in the prediction:

K = 1: The new instance is assigned the same class as its single nearest neighbor. This often leads to overfitting and makes the model highly sensitive to noise.

K = 10: The predicted class is determined by the majority vote of the ten nearest neighbors.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
neigh_cls = KNeighborsClassifier(n_neighbors=3)   # Create the model and choose n neighbors
neigh_cls.fit(X_train, y_train) # Fit the model to the data

In [ ]:
neigh_cls.score(X_test,y_test)  # Calculate the accuracy of the model

In [ ]:
print(classification_report(y_test,neigh_cls.predict(X_test)))

In [ ]:
log_loss(y_test,neigh_cls.predict_proba(X_test))

In [ ]:
RocCurveDisplay.from_estimator(neigh_cls,X_test,y_test)  # Plot the ROC curve
plt.show()

#AUC for coin flip would be 0.5

In [ ]:
PrecisionRecallDisplay.from_estimator(neigh_cls,X_test,y_test)  # Plot the precision-recall curve
plt.show()

Logistic regression

Logistic regression is used to estimate the probability of a binary variable. It is a classification model even though it's called regression.

Linear regression predicts outputs that are not limited to 0 and 1, and is therefore difficult to use for a binary classification task.

Logistic regression draws a boundary between two groups and tells you the probability that a new observation belongs to one of them.


In [ ]:
import sklearn.linear_model as sk_lm
logit_model = sk_lm.LogisticRegression(max_iter=1000)
logit_model.fit(X_train,y_train)

In [ ]:
logit_model.score(X_test,y_test)

In [ ]:
print(classification_report(y_test,logit_model.predict(X_test)))

In [ ]:
log_loss(y_test,logit_model.predict_proba(X_test))

In [ ]:
RocCurveDisplay.from_estimator(logit_model,X_test,y_test)
plt.show()

In [ ]:
PrecisionRecallDisplay.from_estimator(logit_model,X_test,y_test)
plt.show()

Key takeaways?

* Accurate financial variable prediction with a small dataset is hard
* Unbalanced y-category is causing difficulties for the False label

How to try to improve the model?

* Try adding or removing predictive variables
* Standardizing (z-score) variables
* Change the size of training and testing sets (for example 90/10 split)
* Change model parameters

More models are introduced in the course book

# Regression 

* Try a small sample where regularized models are at their best compared to Ordinary Least Squares
* OLS first

**Linear regression**

Estimated Y is a linear combination of x inputs.

From the [coursebook](https://mranta-ai.github.io/data-science-in-finance/ml-for-structured-data/#regression-models):

![](https://mranta-ai.github.io/data-science-in-finance/build/58e44f3f86afb7b0a2b209742c36165c.png)

Main assumptions for linear models:

Linearity:

* Assumption: The relationship between the independent (x) variables and the dependent (y) variable is linear.

* Violation Impact: If the relationship is non-linear, the model will fail to capture the true pattern, leading to poor predictions.

* Solution: Use transformations (e.g., log, polynomial) or consider non-linear models.

Independence:

* Assumption: The residuals (errors) are independent of each other.

* Violation Impact: If residuals are correlated (e.g., in time series data), the model may underestimate uncertainty, leading to incorrect confidence intervals and p-values.

* Solution: Use techniques like autoregressive models or include lagged variables.

Homoscedasticity:

* Assumption: The residuals have constant variance across all levels of the independent variables.

* Violation Impact: If the variance of residuals is not constant (heteroscedasticity), the model may give unequal weight to different observations, affecting the reliability of predictions.

* Solution: Use weighted least squares or transform the dependent variable.

Normality of Residuals:

* Assumption: The residuals are normally distributed, especially for small sample sizes.

* Violation Impact: Non-normal residuals can affect the validity of hypothesis tests and confidence intervals.
* Solution: Transform the dependent variable or use robust regression methods.

No Multicollinearity:

* Assumption: Independent variables are not highly correlated with each other.

* Violation Impact: Multicollinearity can inflate the variance of coefficient estimates, making them unstable and difficult to interpret.

* Solution: Remove or combine correlated variables, or use regularized regression (e.g., ridge or LASSO).

In [ ]:
# Split data into training and test sets using regression features/target
# We will use only 10% of the full data to train the model

X_train, X_test , y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.9, random_state=1)


In [ ]:
len(X_train)

In [ ]:
import sklearn.linear_model as sk_lm

In [ ]:
model = sk_lm.LinearRegression()

In [ ]:
model.fit(X_train,y_train)

In [ ]:
model.coef_

In [ ]:
model.intercept_

In [ ]:
model.score(X_test,y_test)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
#Compare this to the other models
mean_squared_error(y_test,model.predict(X_test))

In [ ]:
plt.style.use('bmh')

from sklearn.metrics import PredictionErrorDisplay
from sklearn.model_selection import LearningCurveDisplay
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from scipy import stats

In [ ]:
#SKIP THIS PLOT DURING LECTURE, LOOK AT YOUR OWN RESPONSIBILITY


fig, axs = plt.subplots(3, 2, figsize=(15, 12))
# Prediction Error Display
PredictionErrorDisplay.from_estimator(model, X_test, y_test, kind='actual_vs_predicted',ax=axs[0, 0])
axs[0, 0].set_title('Prediction Error Display')
# Residual visualization
PredictionErrorDisplay.from_estimator(model, X_test, y_test, kind='residual_vs_predicted',ax=axs[0, 1])
axs[0, 1].set_title('Residual visualization')
# Learning Curve
LearningCurveDisplay.from_estimator(model, X_test, y_test, cv=5, ax=axs[1,0])
axs[1, 0].set_title('Learning Curve')
# QQ Plot for normality of residuals
residuals = y_test - model.predict(X_test)
stats.probplot(residuals, plot=axs[1, 1])
axs[1, 1].set_title('Q-Q Plot')
# Residual histogram
axs[2, 0].hist(residuals, bins=20)
axs[2, 0].axvline(x=0, color='k', linestyle='--')
axs[2, 0].set_title('Residual Distribution')
# Permutation importance
result = permutation_importance(model, X_test, y_test, n_repeats=15, random_state=42)
sorted_idx = result['importances_mean'].argsort()
feature_names = np.array(X_test.columns)
axs[2, 1].barh(feature_names[sorted_idx], result['importances_mean'][sorted_idx])
axs[2, 1].set_title("Permutation Feature Importance")
axs[2, 1].set_xlabel("Mean accuracy decrease")

plt.tight_layout()
plt.subplots_adjust(wspace=0.5, hspace=0.4)

In [ ]:
# Partial Dependence Plots
fig, ax = plt.subplots(figsize=(12, 12))
PartialDependenceDisplay.from_estimator(model, X_test, features=list(X_test.columns),kind='average', ax=ax, n_jobs=-1)
# Add a main title
plt.suptitle("Partial Dependence Analysis", fontsize=20, fontweight='bold', y=1.02)
# Add a subtitle with explanation
fig.text(0.5, 0.98,
         "How each feature affects the model prediction (average effect)",
         ha='center', fontsize=14, fontstyle='italic')
plt.tight_layout()
plt.show()

In [ ]:
# Blue dots visualize the true test observations. Red dots visualize the predictions. Separately for each variable.
fig, axs = plt.subplots(3,3,figsize=(15,12))
for ax,feature in zip(axs.flat,X_test.columns):
    ax.scatter(X_test[feature],y_test,alpha=0.4)
    ax.plot(X_test[feature],model.predict(X_test),'ro',alpha=0.1)
    ax.set_title(feature)
plt.subplots_adjust(hspace=0.2)

Ridge regression


Introduces alpha λ≥0 which is a tuning hyperparameter:

* If λ=0, the objective is identical to OLS regression.

* As λ→∞, the penalty term dominates, and the coefficients approach zero.

By shrinking the coefficients, Ridge regression decreases the variance of the predictions at the expense of a slight increase in bias (the bias-variance tradeoff). Note that the intercept is not included in the regularization penalty, as it represents the mean output when all features are zero.

Ridge regression is particularly useful when:

* There is high multicollinearity among features.

* The number of features is close to or even larger than the number of observations


In [ ]:
ridge = sk_lm.Ridge(alpha=0.001)

In [ ]:
from sklearn.preprocessing import StandardScaler


In [ ]:
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)


In [ ]:
X_train_norm = pd.DataFrame(X_train_norm,columns=X_train.columns)
X_test_norm = pd.DataFrame(X_test_norm,columns=X_test.columns)


In [ ]:
ridge.fit(X_train_norm,y_train)

In [ ]:
ridge.coef_

In [ ]:
ridge.intercept_

In [ ]:
ridge.score(X_test_norm,y_test)

In [ ]:
mean_squared_error(y_test,ridge.predict(X_test_norm))

In [ ]:
fig, axs = plt.subplots(3,3,figsize=(15,12))
for ax,feature in zip(axs.flat,X_test_norm.columns):
    ax.scatter(X_test_norm[feature],y_test,alpha=0.3)
    ax.plot(X_test_norm[feature],ridge.predict(X_test_norm),'r.',alpha=0.3)
    ax.set_title(feature)

Lasso regression

While ridge regression decreases the importance of some features, Lasso regression will remove those features completely that don't improve the model.

In [ ]:
lassocv = sk_lm.LassoCV()

In [ ]:
lassocv.fit(X_train_norm,y_train)

In [ ]:
lassocv.coef_

In [ ]:
lassocv.intercept_

In [ ]:
lassocv.alpha_

In [ ]:
lassocv.score(X_test_norm,y_test)

In [ ]:
mean_squared_error(y_test,lassocv.predict(X_test_norm))

In [ ]:
fig, axs = plt.subplots(3,3,figsize=(15,12))
for ax,feature in zip(axs.flat,X_test_norm.columns):
    ax.scatter(X_test_norm[feature],y_test,alpha=0.3)
    ax.plot(X_test_norm[feature],lassocv.predict(X_test_norm),'r.',alpha=0.3)
    ax.set_title(feature)

Key takeaways?

* ROA prediction using working capital turnover ratios is still difficult
* There are likely some other variables that are affecting ROA more than working capital and the model is missing variables

How to improve the model?

* Variable standardization
* Change the size of training and testing samples
* Add or remove variables
* Hyperparameter tuning